# 厨二病ベクトルの作成（paraphrase-multilingual-MiniLM-L12-v2）

## 1. 文埋め込み（BERT）

In [11]:
!pip -q install -U sentence-transformers

In [12]:
from pathlib import Path

def load_lines(path):
    lines = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

chuni = load_lines("/content/chuni_sentences.txt")
normal = load_lines("/content/normal_sentences.txt")

print("厨二病文数:", len(chuni))
print("普通文数:", len(normal))
print("厨二病サンプル:", chuni[0])
print("普通文サンプル:", normal[0])

厨二病文数: 130
普通文数: 130
厨二病サンプル: 妖精さんたち！私に力を貸して！！！魅惑の香りに飲み込まれなさい"ポイズン・ベリー"！！！
普通文サンプル: コノリジン(Conolidine)は、アルカロイドの1つである


In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 日本語でも比較的安定して使える多言語SBERT
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

E_chu = model.encode(chuni, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
E_nor = model.encode(normal, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

mu_chu = E_chu.mean(axis=0)
mu_nor = E_nor.mean(axis=0)

v_chu = mu_chu - mu_nor
v_chu = v_chu / np.linalg.norm(v_chu)  # 正規化（コサイン用）

np.save("chuni_vector.npy", v_chu)

print("v_chu shape:", v_chu.shape)
print("chuni_vector.npy を保存しました")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

v_chu shape: (384,)
chuni_vector.npy を保存しました


## 2. 厨二病ベクトルの定義

In [14]:
import numpy as np

# ベクトル読み込み
v_chu = np.load("chuni_vector.npy")

# すでに normalize_embeddings=True なので内積＝コサイン類似度
scores_chu = E_chu @ v_chu
scores_nor = E_nor @ v_chu

topk = 10

print("\n=== 厨二病文：上位10（厨二病度が高い） ===")
idx = np.argsort(-scores_chu)[:topk]
for i in idx:
    print(f"{scores_chu[i]:+.3f}  {chuni[i]}")

print("\n=== 普通文：下位10（厨二病度が低い） ===")
idx2 = np.argsort(scores_nor)[:topk]
for i in idx2:
    print(f"{scores_nor[i]:+.3f}  {normal[i]}")


=== 厨二病文：上位10（厨二病度が高い） ===
+0.754  はぁはぁ、行くな！行ってくれるな！！俺を運ぶ鉄の塊、頼む待ってくれっ！！はぁはぁ次遅れたら怒ら..神の逆鱗に触れてしまうんだ！あ...あぁ終わった。
+0.746  俺の心が叫んでいるんだ！最高のエンディングをむかえろと！立て！まだ君はチャレンジしていないんだ！
+0.733  「皆さんどうしました？俺なんかやっちゃいました？俺の力強すぎました？あっはー流石俺っすわ！」
+0.718  「本気を見せる時が来たようだね。一度しか見せないから見逃すなよ？」
+0.708  そんな悲しい運命なんて俺が撃ち砕く。共に行こう。さぁ手を。
+0.704  「中二病ってなんなんだろうな？みんな俺の事中二病って言うけどさ...それが悪いとは思わないんだ俺は。俺の力にひれ伏すがいい！！ってね。中二病ってのは元気が出るんだ」
+0.703  「ここを通りたければ、私を倒してから行くがわい！！！！」くぅ〜！かっこいいぜ！いつか、実際に言いたいセリフトップ3に入るよな！賭ける？いいぞ笑俺らの中で、誰が1番先に言えるか勝負だな！
+0.700  あなたこんな力を持っていながら、何もできないの？そうだわ！私の仲間になるといいわ！
+0.696  これも運命か…俺にはかなり堪えたぞ。ふふふ…み、認めてやろう。お前の能力『未読スルー』を…
+0.680  「振り返らない...僕は進むだけさ。ただまっすぐに君のいない世界を進んでいく。君は早すぎるよ...」

=== 普通文：下位10（厨二病度が低い） ===
-0.308  2017年には、「東京小町」（東京都農林総合研究センター育成）という品種が農林水産省に品種登録された
-0.284  滋賀県（しがけん）は、日本の近畿地方に位置する県
-0.257  地域学部（ちいきがくぶ）とは、人文科学・社会科学・自然科学を複合的に学び、日本国内の地域社会の創造を教育研究するために大学に置かれる学部の名称
-0.227  中国王朝時代の香港の歴史は、秦が統治していた紀元前214年に始まった
-0.224  rsync はディレクトリ内容を表示し、ディレクトリやファイルをコピーできる
-0.219  rsync はローカルなディレクトリ間の同期にも使えるし、RSHやSSHなどのリモートシェ

## 3. 厨二病ベクトルの評価

In [15]:
print("平均スコア")
print("厨二病:", float(scores_chu.mean()))
print("普通  :", float(scores_nor.mean()))

平均スコア
厨二病: 0.5424013733863831
普通  : -0.04448331892490387
